# Phase 7 - Evaluation

Project: ComplaintIQ

Work through this notebook incrementally. Validate each step before proceeding.

In [1]:
from pathlib import Path
PROJECT_ROOT = Path.home() / 'complaintiq'
print('Project root:', PROJECT_ROOT)
print('Exists:', PROJECT_ROOT.exists())

Project root: /home/sagemaker-user/complaintiq
Exists: True


In [2]:
from pathlib import Path
from datetime import datetime, timezone
import os
import sys
import json
import time
import uuid
import sqlite3
import hashlib
import statistics
import traceback

import numpy as np
import pandas as pd

ROOT = Path("/home/sagemaker-user/complaintiq")
DATA = ROOT / "data"
PROC = DATA / "processed"
INDEX = ROOT / "index"
STATE = ROOT / "state"
SRC = ROOT / "src"
EVAL = ROOT / "eval"

EVAL.mkdir(parents=True, exist_ok=True)

GOLD_DIR = EVAL / "gold"
RESULT_DIR = EVAL / "results"

GOLD_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", ROOT)
print("Evaluation:", EVAL)
print("Golden sets:", GOLD_DIR)
print("Results:", RESULT_DIR)

assert ROOT.exists()
assert (PROC / "cfpb_holdout_june_2026.csv").exists()
assert (INDEX / "reg_vectors.npy").exists()
assert (INDEX / "case_vectors.npy").exists()

print("PASS — Evaluation directories and source artifacts verified.")

Project: /home/sagemaker-user/complaintiq
Evaluation: /home/sagemaker-user/complaintiq/eval
Golden sets: /home/sagemaker-user/complaintiq/eval/gold
Results: /home/sagemaker-user/complaintiq/eval/results
PASS — Evaluation directories and source artifacts verified.


In [23]:
from pathlib import Path
import json
import ast
import re

ROOT = Path("/home/sagemaker-user/complaintiq")
NOTEBOOK_DIR = ROOT / "notebooks"

SOURCE_NOTEBOOKS = {
    "Notebook3": NOTEBOOK_DIR / "03_retrieval.ipynb",
    "Notebook4": NOTEBOOK_DIR / "04_agents.ipynb",
    "Notebook5": NOTEBOOK_DIR / "05_graph.ipynb",
}

NOTEBOOK_CATALOG = {}

for notebook_name, notebook_path in SOURCE_NOTEBOOKS.items():

    assert notebook_path.exists(), (
        f"Notebook not found: {notebook_path}"
    )

    notebook = json.loads(
        notebook_path.read_text(encoding="utf-8")
    )

    catalog = []

    for cell_number, cell in enumerate(notebook["cells"]):

        if cell["cell_type"] != "code":
            continue

        source = "".join(cell.get("source", []))

        if not source.strip():
            continue

        # Skip notebook magic commands for catalog parsing
        clean_source = "\n".join(
            line for line in source.splitlines()
            if not line.lstrip().startswith(("%", "!"))
        )

        try:
            tree = ast.parse(clean_source)
        except SyntaxError:
            continue

        definitions = []

        for node in tree.body:

            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                definitions.append(f"function: {node.name}")

            elif isinstance(node, ast.ClassDef):
                definitions.append(f"class: {node.name}")

            elif isinstance(node, (ast.Import, ast.ImportFrom)):
                definitions.append("import")

            elif isinstance(node, (ast.Assign, ast.AnnAssign)):
                targets = (
                    node.targets if isinstance(node, ast.Assign)
                    else [node.target]
                )

                for target in targets:
                    if isinstance(target, ast.Name):
                        definitions.append(f"variable: {target.id}")

        if definitions:
            catalog.append({
                "cell_number": cell_number,
                "definitions": definitions,
                "source": source
            })

    NOTEBOOK_CATALOG[notebook_name] = catalog

    print(f"\n{'=' * 65}")
    print(notebook_name)
    print(f"{'=' * 65}")

    for item in catalog:
        print(
            f"Cell {item['cell_number']:>3}: "
            + ", ".join(item["definitions"])
        )

print("\nPASS — Saved notebook definitions inventoried.")


Notebook3
Cell   1: import, variable: PROJECT_ROOT
Cell   2: import, import, variable: PROJECT_ROOT, import
Cell   3: import, import, import, import, variable: PROJECT, variable: DATA, variable: PROC, variable: INDEX, variable: REPORTS, variable: EVAL, variable: REGION, variable: BUCKET, variable: S3_INDEX
Cell   5: import, import, variable: PROJECT, variable: PROC, variable: INDEX_DIR, variable: S3_INDEX, variable: required, variable: doc_count, variable: pool_df, variable: holdout_df, variable: pool_count, variable: holdout_count, variable: pool_dates, variable: holdout_dates
Cell   7: import, import
Cell   8: import, variable: AWS_REGION, variable: session, variable: brt
Cell   9: variable: test_body, variable: response, variable: test_vector
Cell  11: variable: CACHE, function: cache_path, function: embed_one, function: embed
Cell  12: function: read_jsonl, variable: reg_chunks, variable: chunk_ids
Cell  14: variable: chunk_lengths
Cell  15: import, import, import, import, import,

In [3]:
import boto3

REGION = "us-east-1"
BUCKET = "sagemaker-complaintiq-456427397380"

MODELS = {
    "worker": "us.anthropic.claude-sonnet-4-6",
    "judge": "us.anthropic.claude-opus-4-8",
    "fast": "us.anthropic.claude-sonnet-4-6",
    "embed": "amazon.titan-embed-text-v2:0"
}

session = boto3.Session(region_name=REGION)

s3 = session.client("s3")
brt = session.client("bedrock-runtime")

# Verify S3 access
s3.head_bucket(Bucket=BUCKET)

# Verify Bedrock client configuration
assert brt.meta.region_name == REGION

print("Region:", REGION)
print("Bucket:", BUCKET)
print("Models:", MODELS)
print("PASS — AWS clients restored.")

Region: us-east-1
Bucket: sagemaker-complaintiq-456427397380
Models: {'worker': 'us.anthropic.claude-sonnet-4-6', 'judge': 'us.anthropic.claude-opus-4-8', 'fast': 'us.anthropic.claude-sonnet-4-6', 'embed': 'amazon.titan-embed-text-v2:0'}
PASS — AWS clients restored.


In [25]:
# ============================================================
# COMPLAINTIQ PHASE 7 — NOTEBOOK 6 BOOTSTRAP
# Restore validated implementations from Notebooks 3, 4 and 5
# No corpus embedding generation, benchmark calls or publishing
# ============================================================

import sys
import os
import json
import ast
import uuid
import re
import hashlib
import sqlite3
import importlib
import inspect

from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import boto3
import faiss
import duckdb
import sqlglot

from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi
from pydantic import BaseModel, Field
from typing import Annotated, Literal, TypedDict
import operator

from botocore.exceptions import ClientError
from langgraph.types import interrupt


# ------------------------------------------------------------
# 1. PROJECT CONFIGURATION
# ------------------------------------------------------------

ROOT = Path("/home/sagemaker-user/complaintiq")

DATA = ROOT / "data"
PROC = DATA / "processed"
INDEX = ROOT / "index"
STATE = ROOT / "state"
SRC = ROOT / "src"
EVAL = ROOT / "eval"

for path in [DATA, PROC, INDEX, STATE, SRC, EVAL]:
    path.mkdir(parents=True, exist_ok=True)

REGION = "us-east-1"
AWS_REGION = REGION
BUCKET = "sagemaker-complaintiq-456427397380"

MODELS = {
    "worker": "us.anthropic.claude-sonnet-4-6",
    "judge": "us.anthropic.claude-opus-4-8",
    "fast": "us.anthropic.claude-sonnet-4-6",
    "embed": "amazon.titan-embed-text-v2:0"
}

session = boto3.Session(region_name=REGION)

s3 = session.client("s3")
brt = session.client("bedrock-runtime")

sys.path.insert(0, str(SRC))

print("PASS — Project and AWS configuration initialized.")


# ------------------------------------------------------------
# 2. RESTORE PHASE 4 LLM MODULE
# ------------------------------------------------------------

import llm
importlib.reload(llm)

assert callable(llm.chat)
assert callable(llm.chat_json)
assert callable(llm.get_usage_report)

print("PASS — Phase 4 LLM wrapper restored.")


# ------------------------------------------------------------
# 3. RESTORE PHASE 3 RETRIEVAL FROM NOTEBOOK 5 CELL 19
# ------------------------------------------------------------

NB5_PATH = ROOT / "notebooks" / "05_graph.ipynb"

assert NB5_PATH.exists(), f"Missing notebook: {NB5_PATH}"

with open(NB5_PATH, "r", encoding="utf-8") as f:
    nb5 = json.load(f)

retrieval_source = "".join(
    nb5["cells"][19]["source"]
)

# Execute the validated original retrieval restoration cell.
# It loads saved vectors and reconstructs the indexes.
# It does not regenerate corpus embeddings.

exec(
    compile(
        retrieval_source,
        "notebook5_cell19_retrieval_restore",
        "exec"
    ),
    globals()
)

assert callable(embed_one)
assert callable(embed)
assert callable(tok)
assert callable(HybridIndex)

assert REG_INDEX.faiss.ntotal == 1784
assert CASE_INDEX.faiss.ntotal == 3000

print("PASS — Phase 3 retrieval layer restored.")


# ------------------------------------------------------------
# 4. RESTORE PHASE 5 AGENTS FROM NOTEBOOK 5 CELL 14
# ------------------------------------------------------------

agent_restore_source = "".join(
    nb5["cells"][14]["source"]
)

# This is the existing selective AST-based restoration
# mechanism from Notebook 5.

exec(
    compile(
        agent_restore_source,
        "notebook5_cell14_agent_restore",
        "exec"
    ),
    globals()
)

print("PASS — Phase 5 agent restoration executed.")


# ------------------------------------------------------------
# 5. RESTORE NOTEBOOK 5 ROUTING HELPERS
# ------------------------------------------------------------

routing_source = "".join(
    nb5["cells"][22]["source"]
)

exec(
    compile(
        routing_source,
        "notebook5_cell22_routing",
        "exec"
    ),
    globals()
)

assert callable(after_analyst)
assert callable(after_critic)
assert callable(after_human)

print("PASS — Graph routing helpers restored.")


# ------------------------------------------------------------
# 6. RESTORE PUBLISHING HELPERS
# ------------------------------------------------------------

# Restore S3 publishing configuration.
# This cell only defines configuration and does not publish.

publishing_config_source = "".join(
    nb5["cells"][25]["source"]
)

exec(
    compile(
        publishing_config_source,
        "notebook5_cell25_publish_config",
        "exec"
    ),
    globals()
)

# Restore usage, audit and publishing functions.
# Definitions are loaded; none are invoked.

publishing_helpers_source = "".join(
    nb5["cells"][26]["source"]
)

exec(
    compile(
        publishing_helpers_source,
        "notebook5_cell26_publish_helpers",
        "exec"
    ),
    globals()
)

publishing_node_source = "".join(
    nb5["cells"][30]["source"]
)

exec(
    compile(
        publishing_node_source,
        "notebook5_cell30_publish_node",
        "exec"
    ),
    globals()
)

assert callable(get_usage_snapshot)
assert callable(append_audit_record)
assert callable(publish_node)

print("PASS — Publishing helper definitions restored.")
print("NOTE — No S3 report or audit write was performed.")


# ------------------------------------------------------------
# 7. VALIDATE REQUIRED AGENT FUNCTIONS
# ------------------------------------------------------------

REQUIRED_FUNCTIONS = [
    "guard_node",
    "router_node",
    "triage_node",
    "analyst_node",
    "cases_node",
    "research_node",
    "drafter_node",
    "critic_node",
    "human_node",
    "refuse_node",
    "after_analyst",
    "after_critic",
    "after_human",
    "publish_node",
]

for name in REQUIRED_FUNCTIONS:
    obj = globals().get(name)

    if not callable(obj):
        raise RuntimeError(
            f"Bootstrap validation failed: {name} is missing."
        )

    print(f"PASS | {name}")


# ------------------------------------------------------------
# 8. VALIDATE SHARED STATE AND RETRIEVAL
# ------------------------------------------------------------

assert "CIQState" in globals()
assert "TriageResult" in globals()

assert len(TAXONOMY) > 0

assert REG_INDEX.faiss.ntotal == 1784
assert CASE_INDEX.faiss.ntotal == 3000

assert len(reg_chunks) == 1784
assert len(case_chunks) == 3000

assert reg_vectors.shape == (1784, 1024)
assert case_vectors.shape == (3000, 1024)


# ------------------------------------------------------------
# 9. FINAL BOOTSTRAP REPORT
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("COMPLAINTIQ PHASE 7 BOOTSTRAP VALIDATION")
print("=" * 65)

print("LLM wrapper:          PASS")
print("Embedding functions:  PASS")
print("HybridIndex:           PASS")
print("Regulatory index:     ", REG_INDEX.faiss.ntotal)
print("Complaint index:      ", CASE_INDEX.faiss.ntotal)
print("CIQState:              PASS")
print("Triage taxonomy:      ", len(TAXONOMY))
print("Agent functions:      ", len(REQUIRED_FUNCTIONS))
print("Routing helpers:       PASS")
print("Publishing helpers:    PASS")

print("\nPASS — PHASE 7 RUNTIME RESTORED.")

PASS — Project and AWS configuration initialized.
PASS — Phase 4 LLM wrapper restored.
PASS — All four existing retrieval artifacts found.
Regulatory metadata: 1784
Regulatory vectors: (1784, 1024)
Complaint metadata: 3000
Complaint vectors: (3000, 1024)

PASS — Regulatory index: 1784
PASS — Complaint index: 3000

PASS — Phase 3 retrieval layer restored.
PASS — Phase 3 retrieval layer restored.
Executed implementation cells: [44, 46, 48, 50, 52, 60, 61, 62, 63, 67]
PASS | CIQState
PASS | guard_node
PASS | router_node
PASS | triage_node
PASS | analyst_node
PASS | cases_node
PASS | research_node
PASS | drafter_node
PASS | critic_node
PASS | human_node
PASS | refuse_node
PASS — Phase 5 agent restoration executed.
PASS — All graph-routing functions validated.
PASS — Graph routing helpers restored.
BUCKET: sagemaker-complaintiq-456427397380
REPORT PREFIX: complaintiq/reports
AUDIT KEY: complaintiq/reports/audit.jsonl
PASS — Publishing configuration restored.
PASS — Usage snapshot and audit 

In [26]:
print("CIQState annotations:")
print(CIQState.__annotations__)

CIQState annotations:
{'request': <class 'str'>, 'clean_request': <class 'str'>, 'flags': list[str], 'intent': typing.Literal['TRIAGE', 'ANALYTICS', 'REG_QA', 'BRIEF', 'OUT_OF_SCOPE'], 'triage': <class 'dict'>, 'similar_cases': list[dict], 'sql': <class 'str'>, 'sql_rows': list[dict], 'evidence': list[dict], 'draft': <class 'str'>, 'critique': <class 'dict'>, 'revisions': <class 'int'>, 'human_decision': <class 'dict'>, 'report_uri': <class 'str'>, 'trace': typing.Annotated[list[dict], <built-in function add>]}


In [27]:
def tok(s):
    return re.findall(
        r"[a-z0-9]+(?:\.[0-9]+)?",
        str(s).lower()
    )

In [28]:
print("=== FINAL DEPENDENCY CHECK ===")

checks = [
    ("embed_one", callable(embed_one)),
    ("embed", callable(embed)),
    ("tok", callable(tok)),
    ("HybridIndex", callable(HybridIndex)),
    ("REG_INDEX", REG_INDEX.faiss.ntotal == 1784),
    ("CASE_INDEX", CASE_INDEX.faiss.ntotal == 3000),
    ("CIQState", "CIQState" in globals()),
    ("TriageResult", "TriageResult" in globals()),
    ("llm.chat", callable(llm.chat)),
    ("triage_node", callable(triage_node)),
    ("analyst_node", callable(analyst_node)),
    ("research_node", callable(research_node)),
    ("critic_node", callable(critic_node)),
    ("publish_node", callable(publish_node)),
]

for name, passed in checks:
    print(f"{'PASS' if passed else 'FAIL'} | {name}")

assert all(passed for _, passed in checks)

print("\nALL DEPENDENCY CHECKS PASSED.")

=== FINAL DEPENDENCY CHECK ===
PASS | embed_one
PASS | embed
PASS | tok
PASS | HybridIndex
PASS | REG_INDEX
PASS | CASE_INDEX
PASS | CIQState
PASS | TriageResult
PASS | llm.chat
PASS | triage_node
PASS | analyst_node
PASS | research_node
PASS | critic_node
PASS | publish_node

ALL DEPENDENCY CHECKS PASSED.


In [5]:

import duckdb

# --------------------------------------------------
# Phase 7 — Evaluation database validation
# --------------------------------------------------

DB_PATH = ROOT / "data" / "complaintiq.duckdb"

assert DB_PATH.exists(), f"Database not found: {DB_PATH}"

print("DuckDB:", DB_PATH)

# Close any existing read-only connection from this notebook
if "ro" in globals():
    try:
        ro.close()
    except Exception:
        pass

# Open database in read-only mode
ro = duckdb.connect(str(DB_PATH), read_only=True)

# Verify complaint view
view_check = ro.execute("""
    SELECT COUNT(*)
    FROM v_complaints
""").fetchone()[0]

print("v_complaints rows:", view_check)

assert view_check > 0, "v_complaints is empty"

print("PASS — Read-only evaluation database restored.")


DuckDB: /home/sagemaker-user/complaintiq/data/complaintiq.duckdb
v_complaints rows: 37923
PASS — Read-only evaluation database restored.


In [6]:
sys.path.insert(0, str(SRC))

import importlib
import llm

importlib.reload(llm)

assert callable(llm.chat)
assert callable(llm.get_usage_report)

print(json.dumps(
    llm.get_usage_report(),
    indent=2,
    default=str
))

print("PASS — Phase 4 LLM wrapper restored.")

{
  "ledger": {
    "calls": 0,
    "input_tokens": 0,
    "output_tokens": 0,
    "total_tokens": 0,
    "estimated_cost_usd": 0.0,
    "latency": {
      "calls": 0,
      "average_ms": 0,
      "p50_ms": 0,
      "p95_ms": 0
    },
    "models_used": []
  },
  "cache": {
    "cache_path": "/home/sagemaker-user/complaintiq/state/llm_cache.sqlite",
    "cached_responses": 9
  },
  "models_without_temperature": []
}
PASS — Phase 4 LLM wrapper restored.


In [9]:
class HybridIndex:

    def __init__(self, chunks, vectors):
        self.chunks = list(chunks)
        self.vectors = np.asarray(vectors, dtype="float32")

        if len(self.vectors) != len(self.chunks):
            raise ValueError("Vector/metadata count mismatch")

        if not np.isfinite(self.vectors).all():
            raise ValueError("Invalid vectors")

        self.faiss = faiss.IndexFlatIP(
            self.vectors.shape[1]
        )

        self.faiss.add(self.vectors)

        self.bm25 = BM25Okapi([
            tok(c.get("text", ""))
            for c in self.chunks
        ])

    def search(self, query, k=8, pool=40, rrf_k=60):

        if not query or not str(query).strip():
            raise ValueError("Query cannot be blank")

        pool = min(pool, len(self.chunks))

        qv = embed([query], workers=1)

        _, dense_ids = self.faiss.search(qv, pool)

        sparse_scores = self.bm25.get_scores(tok(query))

        sparse_ids = np.argsort(
            sparse_scores
        )[::-1][:pool]

        scores = defaultdict(float)

        for ranking in (dense_ids[0], sparse_ids):
            for rank, idx in enumerate(ranking):
                if int(idx) >= 0:
                    scores[int(idx)] += (
                        1.0 / (rrf_k + rank + 1)
                    )

        citation = re.search(
            r"\b(\d{1,2})\s*CFR\s*(\d{3,4}\.\d+)\b",
            query,
            re.I
        )

        if citation:
            target = (
                f"{citation.group(1)} CFR "
                f"{citation.group(2)}"
            ).lower()

            compact = (
                f"{citation.group(1)}cfr"
                f"{citation.group(2)}"
            ).lower()

            for idx, rec in enumerate(self.chunks):

                haystack = " ".join([
                    str(rec.get("chunk_id", "")),
                    str(rec.get("section", "")),
                    str(rec.get("text", ""))
                ]).lower()

                normalized = re.sub(
                    r"\s+", " ", haystack
                )

                compact_haystack = re.sub(
                    r"[^a-z0-9.]", "", haystack
                )

                if (
                    target in normalized
                    or compact in compact_haystack
                ):
                    scores[idx] += 1.0

        top = sorted(
            scores,
            key=scores.get,
            reverse=True
        )[:k]

        return [
            {
                **self.chunks[i],
                "rrf": round(float(scores[i]), 6)
            }
            for i in top
        ]



In [11]:
assert "HybridIndex" in globals()
assert callable(HybridIndex)

print("PASS — HybridIndex class restored.")

PASS — HybridIndex class restored.


In [13]:
import re

def tok(text):
    """
    Tokenize text for BM25 retrieval.
    Uses the same lowercase alphanumeric tokenization
    expected by the retrieval index.
    """
    return re.findall(r"\b\w+\b", str(text).lower())

print("PASS — tok() function defined.")
print("Test:", tok("12 CFR 1005.11 Electronic Fund Transfers"))

PASS — tok() function defined.
Test: ['12', 'cfr', '1005', '11', 'electronic', 'fund', 'transfers']


In [14]:
assert callable(tok)
assert callable(HybridIndex)

print("PASS — HybridIndex dependencies available.")

PASS — HybridIndex dependencies available.


In [15]:
import re
import faiss
from collections import defaultdict
from rank_bm25 import BM25Okapi

reg_chunks = [
    json.loads(line)
    for line in (INDEX / "reg_chunks.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
]

case_chunks = [
    json.loads(line)
    for line in (INDEX / "case_chunks.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
]

reg_vectors = np.load(INDEX / "reg_vectors.npy")
case_vectors = np.load(INDEX / "case_vectors.npy")

assert reg_vectors.shape == (1784, 1024)
assert case_vectors.shape == (3000, 1024)

# Re-execute the validated HybridIndex class and embed() implementation
# from Notebook 5's saved source cells before running this cell.

REG_INDEX = HybridIndex(reg_chunks, reg_vectors)
CASE_INDEX = HybridIndex(case_chunks, case_vectors)

assert REG_INDEX.faiss.ntotal == 1784
assert CASE_INDEX.faiss.ntotal == 3000

print("PASS — Phase 3 retrieval indexes reconstructed.")

PASS — Phase 3 retrieval indexes reconstructed.


In [17]:
def embed(texts, workers=3, dim=1024, desc="Embedding"):
    texts = list(texts)

    if not texts:
        return np.empty((0, dim), dtype="float32")

    with ThreadPoolExecutor(max_workers=workers) as executor:
        vectors = list(
            tqdm(
                executor.map(lambda t: embed_one(t, dim), texts),
                total=len(texts),
                desc=desc,
                dynamic_ncols=True
            )
        )

    return np.vstack(vectors).astype("float32")


In [20]:
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
import numpy as np

print("PASS — Embedding dependencies imported.")

PASS — Embedding dependencies imported.


In [29]:
print("=== REGULATORY RETRIEVAL TEST ===")

results = REG_INDEX.search(
    "12 CFR 1005.11 error resolution requirements",
    k=5
)

for i, item in enumerate(results, 1):
    print(
        i,
        item.get("chunk_id"),
        str(item.get("text", ""))[:180]
    )

assert len(results) > 0

print("\nPASS — Regulatory retrieval operational.")

=== REGULATORY RETRIEVAL TEST ===


Embedding: 100%|██████████| 1/1 [00:00<00:00,  3.99it/s]


1 exam_manual_v2_2012_page_0700_chunk_0001 CFPB Consumer Laws and Regulations EFTA An error includes any of the following: • An unauthorized EFT. • An incorrect EFT to or from the consumer’s account. • The omission from a p
2 exam_manual_v2_2012_page_0734_chunk_0001 CFPB Examination Checklist EFTA CFPB Manual V.2 (October 2012) Checklist 9 Yes No NA 35. If the financial institution requires written confirmation of an error within 10 business d
3 exam_manual_v2_2012_page_0699_chunk_0002 an EFT, for example, when the card is used to draw cash advances directly from a credit line, only the error resolution provisions of Regulation Z will apply. • If the unauthorized
4 12CFR1026_1026.13_chunk_0004 ) Promptly reports that the amount or account is in dispute; (ii) Mails or delivers to the consumer (at the same time the report is made) a written notice of the name and address o
5 exam_manual_v2_2012_page_0735_chunk_0001 Yes No NA CFPB Examination Checklist EFTA 39. If a bill ing error occurre

In [30]:
# ============================================================
# PHASE 7 — G1: JUNE HOLDOUT DATASET INSPECTION
# ============================================================

import pandas as pd
import json
from pathlib import Path

# Paths
GOLD_DIR = EVAL / "gold"
RESULTS_DIR = EVAL / "results"
ANALYSIS_DIR = EVAL / "analysis"
MANIFEST_DIR = EVAL / "manifests"

for directory in [
    GOLD_DIR,
    RESULTS_DIR,
    ANALYSIS_DIR,
    MANIFEST_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

HOLDOUT_FILE = PROC / "cfpb_holdout_june_2026.csv"

assert HOLDOUT_FILE.exists(), (
    f"June holdout file not found: {HOLDOUT_FILE}"
)

# Load holdout dataset
holdout_df = pd.read_csv(HOLDOUT_FILE)

print("=" * 65)
print("G1 — JUNE HOLDOUT DATASET INSPECTION")
print("=" * 65)

print("\nDataset shape:")
print(holdout_df.shape)

print("\nAvailable columns:")
for column in holdout_df.columns:
    print(f"- {column}")

print("\nData types:")
print(holdout_df.dtypes)

print("\nFirst five records:")
display(holdout_df.head())

print("\nMissing values:")
display(
    holdout_df.isnull().sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

print("\nDuplicate records:")
print(holdout_df.duplicated().sum())

print("\nPASS — June holdout dataset loaded.")

G1 — JUNE HOLDOUT DATASET INSPECTION

Dataset shape:
(6110, 16)

Available columns:
- Date received
- Product
- Sub-product
- Issue
- Sub-issue
- Consumer complaint narrative
- Company public response
- Company
- State
- ZIP code
- Tags
- Submitted via
- Date sent to company
- Company response to consumer
- Timely response?
- Complaint ID

Data types:
Date received                   object
Product                         object
Sub-product                     object
Issue                           object
Sub-issue                       object
Consumer complaint narrative    object
Company public response         object
Company                         object
State                           object
ZIP code                        object
Tags                            object
Submitted via                   object
Date sent to company            object
Company response to consumer    object
Timely response?                object
Complaint ID                     int64
dtype: object

First f

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
0,2026-06-17,Credit card,General-purpose credit card or charge card,"Advertising and marketing, including promotion...",Didn't receive advertised or promotional terms,I am filing this complaint because Barclays ha...,Company has responded to the consumer and the ...,"CITIBANK, N.A.",MN,554XX,NaN,Web,2026-07-30,Closed with monetary relief,Yes,23300227
1,2026-06-23,Credit card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Credit card company isn't resolving a dispute ...,Despite five separate contacts with Citi betwe...,Company has responded to the consumer and the ...,"CITIBANK, N.A.",MO,65109,NaN,Web,2026-06-23,Closed with monetary relief,Yes,23472102
2,2026-06-25,Credit card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Card was charged for something you did not pur...,"My name is XXXX XXXX, I am a resident of the c...",Company has responded to the consumer and the ...,"BANK OF AMERICA, NATIONAL ASSOCIATION",FL,32824,NaN,Web,2026-06-25,Closed with explanation,Yes,23576641
3,2026-06-02,Credit card,General-purpose credit card or charge card,Getting a credit card,Problem getting a working replacement card,These r proofs they r keeping my money they wo...,Company has responded to the consumer and the ...,WELLS FARGO & COMPANY,AZ,85051,NaN,Web,2026-06-02,Closed with explanation,Yes,22815215
4,2026-06-23,Checking or savings account,Checking account,Managing an account,Problem accessing account,Since my last complaints the company has start...,Company has responded to the consumer and the ...,BMO BANK NATIONAL ASSOCIATION,DE,198XX,NaN,Web,2026-06-23,Closed with monetary relief,Yes,23466116



Missing values:


,missing_count
Tags,4886
Company public response,3420
State,49
ZIP code,42
Sub-issue,7
Issue,0
Date received,0
Product,0
Company,0
Consumer complaint narrative,0



Duplicate records:
0

PASS — June holdout dataset loaded.


In [31]:
required = [
    "ROOT", "BUCKET", "MODELS", "s3", "brt",
    "con", "ro", "llm",
    "REG_INDEX", "CASE_INDEX",
    "CIQState",
    "guard_node", "router_node",
    "triage_node", "analyst_node",
    "cases_node", "research_node",
    "drafter_node", "critic_node",
    "human_node", "refuse_node"
]

missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(f"Missing evaluation dependencies: {missing}")

assert len(reg_chunks) == 1784
assert len(case_chunks) == 3000

print("PASS — Notebook 6 runtime is ready.")

PASS — Notebook 6 runtime is ready.


In [33]:
# ============================================================
# PHASE 7 — G1 CELL 9
# REPRODUCIBLE, PRODUCT-STRATIFIED SAMPLING
# ============================================================

import numpy as np
import pandas as pd

G1_SAMPLE_SIZE = 60
G1_RANDOM_SEED = 20261106

required_columns = [
    "Complaint ID",
    "Product",
    "Issue",
    "Sub-issue",
    "Consumer complaint narrative"
]

missing_columns = [
    c for c in required_columns
    if c not in holdout_df.columns
]

assert not missing_columns, (
    f"Missing required columns: {missing_columns}"
)

# Only use records with a usable narrative.
g1_source = holdout_df.copy()

g1_source["Consumer complaint narrative"] = (
    g1_source["Consumer complaint narrative"]
    .fillna("")
    .astype(str)
    .str.strip()
)

g1_source = g1_source[
    g1_source["Consumer complaint narrative"].str.len() > 0
].copy()

# Complaint IDs must be unique.
assert g1_source["Complaint ID"].is_unique, (
    "Duplicate Complaint IDs detected in source data."
)

assert len(g1_source) >= G1_SAMPLE_SIZE

# Product-level proportional allocation using largest remainder.
product_counts = (
    g1_source.groupby("Product")
    .size()
    .sort_index()
)

raw_allocations = (
    product_counts / product_counts.sum()
) * G1_SAMPLE_SIZE

allocations = np.floor(raw_allocations).astype(int)

remaining = G1_SAMPLE_SIZE - allocations.sum()

remainders = (
    raw_allocations - allocations
).sort_values(ascending=False)

for product in remainders.index:
    if remaining == 0:
        break

    if allocations[product] < product_counts[product]:
        allocations[product] += 1
        remaining -= 1

assert allocations.sum() == G1_SAMPLE_SIZE

# Sample independently within each product.
sampled_groups = []

for product, sample_count in allocations.items():
    if sample_count == 0:
        continue

    group = g1_source[
        g1_source["Product"] == product
    ]

    sampled_groups.append(
        group.sample(
            n=int(sample_count),
            random_state=G1_RANDOM_SEED
        )
    )

g1_sample = (
    pd.concat(sampled_groups)
    .sample(
        frac=1,
        random_state=G1_RANDOM_SEED
    )
    .reset_index(drop=True)
)

assert len(g1_sample) == G1_SAMPLE_SIZE
assert g1_sample["Complaint ID"].is_unique

print("=" * 65)
print("G1 — STRATIFIED SAMPLE")
print("=" * 65)

print("Sample size:", len(g1_sample))
print("Random seed:", G1_RANDOM_SEED)

print("\nProduct allocation:")
display(
    pd.DataFrame({
        "source_count": product_counts,
        "sample_count": allocations
    }).fillna(0).astype(int)
)

print("\nSample distribution:")
display(
    g1_sample["Product"]
    .value_counts()
    .to_frame("sample_count")
)

print("\nIssue distribution:")
display(
    g1_sample["Issue"]
    .value_counts()
    .to_frame("sample_count")
)

print("\nPASS — 60 reproducible G1 records selected.")

G1 — STRATIFIED SAMPLE
Sample size: 60
Random seed: 20261106

Product allocation:


,source_count,sample_count
Product,,
Checking or savings account,3270,32
Credit card,2840,28



Sample distribution:


,sample_count
Product,
Checking or savings account,32
Credit card,28



Issue distribution:


,sample_count
Issue,
Managing an account,18
Fees or interest,6
Problem with a purchase shown on your statement,6
Problem with a lender or other company charging your account,5
Closing an account,5
"Other features, terms, or problems",4
Closing your account,3
Improper use of your report,3
Opening an account,2



PASS — 60 reproducible G1 records selected.


In [34]:
# ============================================================
# PHASE 7 — G1 CELL 10
# BUILD GOLD EVALUATION RECORDS
# ============================================================

import json
import pandas as pd

def clean_gold_value(value):
    if pd.isna(value):
        return None

    value = str(value).strip()

    if value == "":
        return None

    return value


g1_gold_records = []

for _, row in g1_sample.iterrows():

    record = {
        "case_id": str(int(row["Complaint ID"])),

        "request": clean_gold_value(
            row["Consumer complaint narrative"]
        ),

        "expected": {
            "product": clean_gold_value(row["Product"]),
            "issue": clean_gold_value(row["Issue"]),
            "sub_issue": clean_gold_value(row["Sub-issue"])
        },

        # These fields are not supplied as ground truth
        # by the CFPB dataset. They require human annotation.
        "manual_annotation": {
            "severity": None,
            "vulnerable_customer": None,
            "vulnerability_signals": None,
            "regulation_hints": None,
            "summary": None
        },

        "source": {
            "dataset": "cfpb_holdout_june_2026",
            "complaint_id": str(int(row["Complaint ID"])),
            "date_received": clean_gold_value(
                row["Date received"]
            )
        },

        "annotation_status": "pending"
    }

    g1_gold_records.append(record)

assert len(g1_gold_records) == 60

print("=" * 65)
print("G1 — GOLD RECORD GENERATION")
print("=" * 65)

print("Records generated:", len(g1_gold_records))

print("\nExample gold record:")
print(
    json.dumps(
        g1_gold_records[0],
        indent=2,
        ensure_ascii=False
    )
)

print("\nPASS — G1 gold records generated.")

G1 — GOLD RECORD GENERATION
Records generated: 60

Example gold record:
{
  "case_id": "23266147",
  "request": "I am filing this complaint against BMO Bank regarding an unreasonable hold placed on my deposited insurance settlement funds and the overdraft fees that resulted from the banks actions. On XX/XX/year>, I deposited a settlement check from XXXX XXXX at the BMO branch located at XXXX XXXX XXXX, XXXX XXXX, Missouri. BMO placed a hold on the funds but failed to provide adequate notice explaining the reason for the hold, the length of the hold, or my rights under the banks funds availability policy. After the deposit, documentation was provided to verify the legitimacy of the settlement funds. The branch also received verification directly from XXXX XXXX confirming that the check was valid and legitimate. Despite receiving this verification, BMO continued to restrict access to my funds. Because BMO withheld access to verified funds, transactions that would have otherwise been cove

In [35]:
# ============================================================
# PHASE 7 — G1 CELL 11
# HUMAN ANNOTATION TEMPLATE
# ============================================================

import pandas as pd

annotation_rows = []

for record in g1_gold_records:

    annotation_rows.append({
        "case_id": record["case_id"],

        "severity": "",
        "vulnerable_customer": "",
        "vulnerability_signals": "",
        "regulation_hints": "",
        "summary": "",

        "reviewer": "",
        "review_notes": "",
        "annotation_status": "PENDING"
    })

g1_annotation_df = pd.DataFrame(annotation_rows)

G1_ANNOTATION_FILE = (
    GOLD_DIR / "G1_triage_manual_annotations.csv"
)

print("=" * 65)
print("G1 — HUMAN ANNOTATION TEMPLATE")
print("=" * 65)

print("Annotation records:", len(g1_annotation_df))

display(g1_annotation_df.head(10))

print("\nSeverity values to use:")
print(["LOW", "MEDIUM", "HIGH", "CRITICAL"])

print("\nVulnerable customer values to use:")
print(["True", "False"])

print("\nAnnotation file target:")
print(G1_ANNOTATION_FILE)

print("\nPASS — Annotation template prepared.")

G1 — HUMAN ANNOTATION TEMPLATE
Annotation records: 60


,case_id,severity,vulnerable_customer,vulnerability_signals,regulation_hints,summary,reviewer,review_notes,annotation_status
0,23266147,,,,,,,,PENDING
1,23172941,,,,,,,,PENDING
2,23492548,,,,,,,,PENDING
3,23304250,,,,,,,,PENDING
4,22965626,,,,,,,,PENDING
5,23082821,,,,,,,,PENDING
6,23494747,,,,,,,,PENDING
7,23053378,,,,,,,,PENDING
8,22988407,,,,,,,,PENDING
9,22794522,,,,,,,,PENDING



Severity values to use:
['LOW', 'MEDIUM', 'HIGH', 'CRITICAL']

Vulnerable customer values to use:
['True', 'False']

Annotation file target:
/home/sagemaker-user/complaintiq/eval/gold/G1_triage_manual_annotations.csv

PASS — Annotation template prepared.


In [36]:
# ============================================================
# PHASE 7 — G1 CELL 12
# SAVE GOLD DATASET
# ============================================================

import json
import hashlib
from pathlib import Path

G1_GOLD_FILE = GOLD_DIR / "G1_triage_gold.jsonl"

# Write JSONL.
with open(G1_GOLD_FILE, "w", encoding="utf-8") as f:

    for record in g1_gold_records:
        f.write(
            json.dumps(
                record,
                ensure_ascii=False
            ) + "\n"
        )

# Save human annotation template.
g1_annotation_df.to_csv(
    G1_ANNOTATION_FILE,
    index=False
)

# Calculate SHA256 checksums.
def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)

    return digest.hexdigest()


print("=" * 65)
print("G1 — SAVED ARTIFACTS")
print("=" * 65)

print("Gold dataset:")
print(G1_GOLD_FILE)
print("Size:", G1_GOLD_FILE.stat().st_size, "bytes")
print("SHA256:", sha256_file(G1_GOLD_FILE))

print("\nAnnotation template:")
print(G1_ANNOTATION_FILE)
print("Size:", G1_ANNOTATION_FILE.stat().st_size, "bytes")
print("SHA256:", sha256_file(G1_ANNOTATION_FILE))

print("\nPASS — G1 artifacts saved locally.")

G1 — SAVED ARTIFACTS
Gold dataset:
/home/sagemaker-user/complaintiq/eval/gold/G1_triage_gold.jsonl
Size: 116918 bytes
SHA256: f6f405dd741175eb8f5e0aa0eb901eedb34a0c633311d601f9d6299d69420984

Annotation template:
/home/sagemaker-user/complaintiq/eval/gold/G1_triage_manual_annotations.csv
Size: 1564 bytes
SHA256: 11452cfbde9e47de87328aab8a6913cff137174d3822796dc23f561604780987

PASS — G1 artifacts saved locally.


In [37]:
# ============================================================
# PHASE 7 — G1 CELL 13
# GOLD DATASET ACCEPTANCE VALIDATION
# ============================================================

import json
import pandas as pd

# Reload saved artifacts instead of validating only memory objects.
with open(G1_GOLD_FILE, encoding="utf-8") as f:
    g1_reloaded = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

g1_annotations = pd.read_csv(
    G1_ANNOTATION_FILE,
    dtype=str
)

# 1. Record count.
assert len(g1_reloaded) == 60

# 2. Unique complaint IDs.
case_ids = [
    record["case_id"]
    for record in g1_reloaded
]

assert len(case_ids) == len(set(case_ids))

# 3. Required input and expected labels.
for record in g1_reloaded:

    assert record["request"].strip()

    assert record["expected"]["product"]
    assert record["expected"]["issue"]

    assert record["source"]["dataset"] == (
        "cfpb_holdout_june_2026"
    )

# 4. Annotation template integrity.
assert len(g1_annotations) == 60
assert g1_annotations["case_id"].is_unique

assert set(g1_annotations["case_id"]) == set(case_ids)

# 5. Verify all IDs belong to the June holdout.
holdout_ids = set(
    holdout_df["Complaint ID"].astype(str)
)

assert set(case_ids).issubset(holdout_ids)

# 6. Check whether complaint identifiers are exposed
# in the existing complaint retrieval metadata.
index_ids = set()

for chunk in CASE_INDEX.chunks:

    possible_ids = [
        chunk.get("complaint_id"),
        chunk.get("Complaint ID"),
        chunk.get("case_id")
    ]

    for value in possible_ids:
        if value is not None:
            index_ids.add(str(value))

overlap = set(case_ids).intersection(index_ids)

print("=" * 65)
print("COMPLAINTIQ — G1 ACCEPTANCE VALIDATION")
print("=" * 65)

print("Gold records:", len(g1_reloaded))
print("Unique IDs:", len(set(case_ids)))
print("Valid narratives:", sum(
    bool(r["request"].strip())
    for r in g1_reloaded
))
print("Valid product labels:", sum(
    bool(r["expected"]["product"])
    for r in g1_reloaded
))
print("Valid issue labels:", sum(
    bool(r["expected"]["issue"])
    for r in g1_reloaded
))
print("Annotation rows:", len(g1_annotations))

print("\nExisting retrieval metadata ID overlap:", len(overlap))

if overlap:
    print("WARNING — Matching IDs found in retrieval metadata.")
    print("Sample matching IDs:", sorted(overlap)[:10])
else:
    print(
        "No matching complaint IDs found in the "
        "explicit metadata ID fields."
    )

print("\nArtifact checksums:")
print("Gold:", sha256_file(G1_GOLD_FILE))
print("Annotations:", sha256_file(G1_ANNOTATION_FILE))

print("\nPASS — G1 GOLD DATASET STRUCTURAL ACCEPTANCE.")

COMPLAINTIQ — G1 ACCEPTANCE VALIDATION
Gold records: 60
Unique IDs: 60
Valid narratives: 60
Valid product labels: 60
Valid issue labels: 60
Annotation rows: 60

Existing retrieval metadata ID overlap: 0
No matching complaint IDs found in the explicit metadata ID fields.

Artifact checksums:
Gold: f6f405dd741175eb8f5e0aa0eb901eedb34a0c633311d601f9d6299d69420984
Annotations: 11452cfbde9e47de87328aab8a6913cff137174d3822796dc23f561604780987

PASS — G1 GOLD DATASET STRUCTURAL ACCEPTANCE.


In [38]:
# ============================================================
# PHASE 7 — RETRIEVAL TOKENIZER CONSISTENCY FIX
# ============================================================

import re
from collections import defaultdict
from rank_bm25 import BM25Okapi

# Restore the original Phase 3 tokenizer.
def tok(s):
    return re.findall(
        r"[a-z0-9]+(?:\.[0-9]+)?",
        str(s).lower()
    )

# Rebuild in-memory indexes using existing artifacts.
REG_INDEX = HybridIndex(reg_chunks, reg_vectors)
CASE_INDEX = HybridIndex(case_chunks, case_vectors)

# Validate the tokenizer.
test_tokens = tok("12 CFR 1005.11 Electronic Fund Transfers")

assert "1005.11" in test_tokens, (
    f"Unexpected tokenization: {test_tokens}"
)

assert REG_INDEX.faiss.ntotal == 1784
assert CASE_INDEX.faiss.ntotal == 3000

print("Tokenizer test:", test_tokens)
print("Regulatory index:", REG_INDEX.faiss.ntotal)
print("Complaint index:", CASE_INDEX.faiss.ntotal)

# Test regulatory retrieval.
results = REG_INDEX.search(
    "12 CFR 1005.11 error resolution requirements",
    k=5
)

assert len(results) == 5

print("\nTop regulatory retrieval results:")
for rank, result in enumerate(results, 1):
    print(
        rank,
        result.get("chunk_id"),
        result.get("rrf")
    )

print("\nPASS — Retrieval tokenizer consistency restored.")

Tokenizer test: ['12', 'cfr', '1005.11', 'electronic', 'fund', 'transfers']
Regulatory index: 1784
Complaint index: 3000


Embedding: 100%|██████████| 1/1 [00:00<00:00, 150.02it/s]



Top regulatory retrieval results:
1 exam_manual_v2_2012_page_0700_chunk_0001 1.032522
2 exam_manual_v2_2012_page_0734_chunk_0001 1.028986
3 exam_manual_v2_2012_page_0699_chunk_0002 1.02885
4 12CFR1026_1026.13_chunk_0004 1.028405
5 exam_manual_v2_2012_page_0735_chunk_0001 1.028283

PASS — Retrieval tokenizer consistency restored.
